**Percepción de los momentos de la simulación clínica y su relación con la confianza y claridad en la priorización de cuidados**

Notebook de Google Colab listo para ejecutar y compartir. Está pensado como tutorial guiado para profesionales de salud, educación o intervención social sin experiencia en programación. Todo el análisis usa datos sintéticos generados dentro del propio cuaderno, así que no hace falta cargar archivos ni escribir código.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', palette='Blues')
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format

def redondear_escala(x):
    return np.clip(np.rint(x), 1, 5).astype(float)


## 1. El estudio en una página

Este ejercicio representa un estudio observacional breve tras una sesión de simulación clínica. La meta no es demostrar efectos, sino practicar una lectura sencilla de relaciones entre variables de percepción.

**Microtarea 1. Anticipación**  
Antes de mirar la tabla, piensa qué momento de la simulación crees que tenderá a recibir mejor valoración: escenario, repetición o debriefing.

In [ ]:
estudio_resumen = pd.DataFrame(
    {
        'Elemento': [
            'Contexto profesional',
            'Pregunta de investigación',
            'Diseño',
            'Unidad de análisis',
            'Resultados principales',
            'Explicativas principales',
            'Variables de contexto',
            'Lectura correcta'
        ],
        'Descripción': [
            'Estudiantes de tercer curso de enfermería en una sesión de simulación clínica.',
            '¿Los estudiantes que valoran positivamente todos los momentos de la simulación clínica presentan mayor confianza y claridad al priorizar cuidados?',
            'Estudio observacional transversal con cuestionario post-simulación y datos sintéticos.',
            'Cada estudiante en una simulación concreta.',
            'Confianza percibida al priorizar cuidados y claridad percibida del razonamiento (1-5).',
            'Valoración del escenario, repetición y debriefing; además, alta valoración global del proceso.',
            'Experiencia previa en simulación y rendimiento académico percibido.',
            'Las diferencias se interpretan como asociación, no como causalidad.'
        ]
    }
)

display(estudio_resumen)

**Microtarea 2. Interpretación**  
Revisa la fila de diseño y localiza la palabra o idea que obliga a hablar de asociación y no de efecto causal.

**Microtarea 3. Crítica o transferencia**  
Si trasladaras este estudio a tu contexto profesional, ¿qué adaptarías primero: la pregunta, la población o el momento de recogida de datos?

## 2. Variables y plan de datos

Trabajaremos con dos resultados y varias variables explicativas sencillas. El objetivo didáctico es practicar una comparación clara entre estudiantes con valoración global alta del proceso y estudiantes con valoración no alta.

**Microtarea 1. Anticipación**  
Antes de ver el plan, decide qué variable esperas que se acerque más a la claridad del razonamiento: escenario, repetición o debriefing.

In [ ]:
variables = pd.DataFrame(
    [
        ['confianza_priorizar', 'Resultado', 'Confianza percibida al priorizar cuidados (1-5)'],
        ['claridad_razonamiento', 'Resultado', 'Claridad percibida del razonamiento (1-5)'],
        ['valoracion_escenario', 'Explicativa', 'Valoración del escenario de simulación (1-5)'],
        ['valoracion_repeticion', 'Explicativa', 'Valoración de la repetición (1-5)'],
        ['valoracion_debriefing', 'Explicativa', 'Valoración del debriefing (1-5)'],
        ['alta_valoracion_global', 'Derivada', 'Grupo que puntúa 4 o 5 en los tres momentos'],
        ['experiencia_previa_simulacion', 'Contexto', 'Ninguna, poca o alta'],
        ['rendimiento_percibido', 'Contexto', 'Bajo, medio o alto']
    ],
    columns=['Variable', 'Tipo', 'Descripción']
)

plan_analisis = pd.DataFrame(
    {
        'Paso': ['1', '2', '3'],
        'Qué haremos': [
            'Describir distribuciones, medias y frecuencias.',
            'Comparar cómo se valoran escenario, repetición y debriefing.',
            'Comparar confianza y claridad entre alta valoración global y valoración no alta.'
        ]
    }
)

display(variables)
display(plan_analisis)

**Microtarea 2. Interpretación**  
Observa la variable derivada. ¿Qué ventaja tiene resumir tres valoraciones en un grupo de comparación sencillo?

**Microtarea 3. Crítica o transferencia**  
¿Añadirías alguna variable de contexto más en un estudio real, por ejemplo edad, turno clínico o familiaridad con la asignatura?

## 3. Generación de datos sintéticos

Ahora generamos datos sintéticos con una función pedagógica clara: crear un patrón principal interpretable, pero sin perfección. La mayoría valorará positivamente la simulación, el debriefing tenderá a quedar algo mejor valorado y habrá solapamiento entre grupos, ruido individual y algunos valores perdidos.

**Microtarea 1. Anticipación**  
Antes de ejecutar la celda, piensa si prefieres un patrón muy limpio o uno algo imperfecto para aprender a interpretar resultados con más realismo.

In [ ]:
rng = np.random.default_rng(42)
n = 160

experiencia = rng.choice(['ninguna', 'poca', 'alta'], size=n, p=[0.28, 0.50, 0.22])
rendimiento = rng.choice(['bajo', 'medio', 'alto'], size=n, p=[0.18, 0.57, 0.25])

efecto_exp = pd.Series(experiencia).map({'ninguna': -0.18, 'poca': 0.00, 'alta': 0.22}).to_numpy()
efecto_rend = pd.Series(rendimiento).map({'bajo': -0.15, 'medio': 0.00, 'alto': 0.18}).to_numpy()

base_valoracion = 3.90 + efecto_exp + 0.50 * efecto_rend + rng.normal(0, 0.35, n)

valoracion_escenario = redondear_escala(base_valoracion - 0.10 + rng.normal(0, 0.65, n))
valoracion_repeticion = redondear_escala(base_valoracion + rng.normal(0, 0.70, n))
valoracion_debriefing = redondear_escala(base_valoracion + 0.25 + rng.normal(0, 0.55, n))

media_momentos = (valoracion_escenario + valoracion_repeticion + valoracion_debriefing) / 3

confianza_priorizar = redondear_escala(
    3.25 + 0.55 * (media_momentos - 3.80) + 0.20 * efecto_exp + 0.10 * efecto_rend + rng.normal(0, 0.75, n)
)

claridad_razonamiento = redondear_escala(
    3.15 + 0.60 * (media_momentos - 3.80) + 0.18 * (valoracion_debriefing - valoracion_escenario)
    + 0.08 * efecto_exp + 0.15 * efecto_rend + rng.normal(0, 0.70, n)
)

df = pd.DataFrame(
    {
        'valoracion_escenario': valoracion_escenario,
        'valoracion_repeticion': valoracion_repeticion,
        'valoracion_debriefing': valoracion_debriefing,
        'confianza_priorizar': confianza_priorizar,
        'claridad_razonamiento': claridad_razonamiento,
        'experiencia_previa_simulacion': experiencia,
        'rendimiento_percibido': rendimiento
    }
)

for columna, cantidad in {
    'valoracion_escenario': 4,
    'valoracion_repeticion': 5,
    'confianza_priorizar': 3,
    'claridad_razonamiento': 4
}.items():
    indices = rng.choice(df.index[df[columna].notna()], size=cantidad, replace=False)
    df.loc[indices, columna] = np.nan

df['alta_valoracion_global'] = np.where(
    df[['valoracion_escenario', 'valoracion_repeticion', 'valoracion_debriefing']].ge(4).all(axis=1),
    'Alta valoración global',
    'Valoración no alta'
)

df['media_momentos'] = df[['valoracion_escenario', 'valoracion_repeticion', 'valoracion_debriefing']].mean(axis=1)

resumen_generacion = pd.DataFrame(
    {
        'Indicador': [
            'Número de estudiantes simulados',
            'Porcentaje con alta valoración global',
            'Media escenario',
            'Media repetición',
            'Media debriefing',
            'Valores perdidos totales'
        ],
        'Valor': [
            len(df),
            round((df['alta_valoracion_global'] == 'Alta valoración global').mean() * 100, 1),
            round(df['valoracion_escenario'].mean(), 2),
            round(df['valoracion_repeticion'].mean(), 2),
            round(df['valoracion_debriefing'].mean(), 2),
            int(df.isna().sum().sum())
        ]
    }
)

display(resumen_generacion)
display(df.head(8))

**Microtarea 2. Interpretación**  
Fíjate en las medias de los tres momentos. ¿Aparece el patrón esperado de una valoración algo más alta en el debriefing?

**Microtarea 3. Crítica o transferencia**  
Valora si este nivel de imperfección te parece útil. ¿Aprenderías más con datos totalmente ordenados o con este pequeño grado de ruido y valores perdidos?

## 4. Inspección inicial

Antes de analizar, conviene revisar tamaño, estructura y valores perdidos. Esta inspección evita interpretar resultados sin saber si las variables están razonablemente completas.

**Microtarea 1. Anticipación**  
Antes de ver los resúmenes, estima si habrá pocos o bastantes valores perdidos y si eso puede cambiar mucho la lectura general.

In [ ]:
resumen_inicial = pd.DataFrame(
    {
        'Indicador': ['Filas', 'Columnas', 'Casos con algún valor perdido', 'Casos completos en variables de resultado'],
        'Valor': [
            df.shape[0],
            df.shape[1],
            int(df.isna().any(axis=1).sum()),
            int(df[['confianza_priorizar', 'claridad_razonamiento']].notna().all(axis=1).sum())
        ]
    }
)

faltantes = df.isna().sum().reset_index()
faltantes.columns = ['Variable', 'Valores perdidos']
faltantes['Porcentaje'] = (faltantes['Valores perdidos'] / len(df) * 100).round(1)

display(resumen_inicial)
display(faltantes[faltantes['Valores perdidos'] > 0])
display(df.sample(6, random_state=7))

**Microtarea 2. Interpretación**  
Observa si los valores perdidos parecen puntuales o masivos. ¿Dirías que permiten seguir analizando sin cambiar por completo el ejercicio?

**Microtarea 3. Crítica o transferencia**  
En un estudio real, ¿qué harías primero ante valores perdidos: revisar el cuestionario, mejorar la recogida o decidir una regla de análisis?

## 5. Análisis descriptivo

Ahora describimos el conjunto de datos. Buscamos una imagen general: cómo son las distribuciones, qué momento recibe mejor valoración y cómo se comportan algunas variables de contexto.

**Microtarea 1. Anticipación**  
Antes de ejecutar la celda, decide si esperas distribuciones muy concentradas en 4 y 5 o más repartidas entre 3, 4 y 5.

In [ ]:
resumen_descriptivo = df[
    ['valoracion_escenario', 'valoracion_repeticion', 'valoracion_debriefing', 'confianza_priorizar', 'claridad_razonamiento']
].agg(['mean', 'std', 'min', 'max']).T.reset_index()
resumen_descriptivo.columns = ['Variable', 'Media', 'Desviación típica', 'Mínimo', 'Máximo']

frecuencia_experiencia = df['experiencia_previa_simulacion'].value_counts().reindex(['ninguna', 'poca', 'alta']).reset_index()
frecuencia_experiencia.columns = ['Experiencia previa', 'Frecuencia']

frecuencia_rendimiento = df['rendimiento_percibido'].value_counts().reindex(['bajo', 'medio', 'alto']).reset_index()
frecuencia_rendimiento.columns = ['Rendimiento percibido', 'Frecuencia']

display(resumen_descriptivo.round(2))
display(frecuencia_experiencia)
display(frecuencia_rendimiento)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

bins = np.arange(0.5, 6.6, 1)
sns.histplot(df['confianza_priorizar'].dropna(), bins=bins, ax=axes[0, 0], color='#4C78A8')
axes[0, 0].set_title('Confianza al priorizar')
axes[0, 0].set_xlabel('Puntuación')

sns.histplot(df['claridad_razonamiento'].dropna(), bins=bins, ax=axes[0, 1], color='#72B7B2')
axes[0, 1].set_title('Claridad del razonamiento')
axes[0, 1].set_xlabel('Puntuación')

medias_momentos = pd.DataFrame(
    {
        'Momento': ['Escenario', 'Repetición', 'Debriefing'],
        'Media': [
            df['valoracion_escenario'].mean(),
            df['valoracion_repeticion'].mean(),
            df['valoracion_debriefing'].mean()
        ]
    }
)
sns.barplot(data=medias_momentos, x='Momento', y='Media', ax=axes[0, 2], palette=['#9ecae1', '#6baed6', '#3182bd'])
axes[0, 2].set_title('Valoración media de los momentos')
axes[0, 2].set_ylim(0, 5)

confianza_contexto = df.groupby('experiencia_previa_simulacion', as_index=False)['confianza_priorizar'].mean()
confianza_contexto['experiencia_previa_simulacion'] = pd.Categorical(
    confianza_contexto['experiencia_previa_simulacion'],
    categories=['ninguna', 'poca', 'alta'],
    ordered=True
)
confianza_contexto = confianza_contexto.sort_values('experiencia_previa_simulacion')
sns.barplot(data=confianza_contexto, x='experiencia_previa_simulacion', y='confianza_priorizar', ax=axes[1, 0], color='#86BCB6')
axes[1, 0].set_title('Confianza media por experiencia previa')
axes[1, 0].set_xlabel('Experiencia previa')
axes[1, 0].set_ylabel('Media')
axes[1, 0].set_ylim(0, 5)

claridad_contexto = df.groupby('rendimiento_percibido', as_index=False)['claridad_razonamiento'].mean()
claridad_contexto['rendimiento_percibido'] = pd.Categorical(
    claridad_contexto['rendimiento_percibido'],
    categories=['bajo', 'medio', 'alto'],
    ordered=True
)
claridad_contexto = claridad_contexto.sort_values('rendimiento_percibido')
sns.barplot(data=claridad_contexto, x='rendimiento_percibido', y='claridad_razonamiento', ax=axes[1, 1], color='#4C9F70')
axes[1, 1].set_title('Claridad media por rendimiento percibido')
axes[1, 1].set_xlabel('Rendimiento percibido')
axes[1, 1].set_ylabel('Media')
axes[1, 1].set_ylim(0, 5)

conteo_grupos = df['alta_valoracion_global'].value_counts().reindex(['Alta valoración global', 'Valoración no alta']).reset_index()
conteo_grupos.columns = ['Grupo', 'Frecuencia']
sns.barplot(data=conteo_grupos, x='Grupo', y='Frecuencia', ax=axes[1, 2], palette=['#2171b5', '#9ecae1'])
axes[1, 2].set_title('Tamaño de los grupos de comparación')
axes[1, 2].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.show()

**Microtarea 2. Interpretación**  
Observa si el patrón general combina dos cosas a la vez: valoraciones altas en conjunto y variabilidad suficiente para que no todos los estudiantes parezcan iguales.

**Microtarea 3. Crítica o transferencia**  
¿Qué gráfico te resulta más útil para una reunión docente rápida: histogramas, barras de medias o la comparación por grupos? Justifica tu elección.

## 6. Análisis principal

Pasamos al contraste central del ejercicio: comparar la confianza y la claridad entre quienes muestran alta valoración global del proceso y quienes no la muestran. La idea es mirar diferencias visibles sin convertirlas en afirmaciones causales.

**Microtarea 1. Anticipación**  
Antes de ejecutar la celda, predice si la diferencia entre grupos será pequeña, moderada o muy grande.

In [ ]:
orden_grupos = ['Alta valoración global', 'Valoración no alta']

comparacion = df.groupby('alta_valoracion_global').agg(
    n=('alta_valoracion_global', 'size'),
    confianza_media=('confianza_priorizar', 'mean'),
    claridad_media=('claridad_razonamiento', 'mean')
).reindex(orden_grupos)

diferencias = pd.DataFrame(
    {
        'Resultado': ['Confianza percibida', 'Claridad percibida'],
        'Media en alta valoración global': [
            comparacion.loc['Alta valoración global', 'confianza_media'],
            comparacion.loc['Alta valoración global', 'claridad_media']
        ],
        'Media en valoración no alta': [
            comparacion.loc['Valoración no alta', 'confianza_media'],
            comparacion.loc['Valoración no alta', 'claridad_media']
        ]
    }
)
diferencias['Diferencia absoluta'] = (
    diferencias['Media en alta valoración global'] - diferencias['Media en valoración no alta']
)

display(comparacion.round(2))
display(diferencias.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(
    data=df,
    x='alta_valoracion_global',
    y='confianza_priorizar',
    order=orden_grupos,
    ax=axes[0],
    showfliers=False,
    palette=['#2171b5', '#9ecae1']
)
sns.stripplot(
    data=df,
    x='alta_valoracion_global',
    y='confianza_priorizar',
    order=orden_grupos,
    ax=axes[0],
    color='0.25',
    alpha=0.35,
    jitter=0.18
)
axes[0].set_title('Confianza según valoración global')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=10)

sns.boxplot(
    data=df,
    x='alta_valoracion_global',
    y='claridad_razonamiento',
    order=orden_grupos,
    ax=axes[1],
    showfliers=False,
    palette=['#2171b5', '#9ecae1']
)
sns.stripplot(
    data=df,
    x='alta_valoracion_global',
    y='claridad_razonamiento',
    order=orden_grupos,
    ax=axes[1],
    color='0.25',
    alpha=0.35,
    jitter=0.18
)
axes[1].set_title('Claridad según valoración global')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.show()

**Microtarea 2. Interpretación**  
Mira la tabla y los boxplots a la vez. ¿Ves una diferencia visible entre grupos sin que desaparezca el solapamiento entre estudiantes?

**Microtarea 3. Crítica o transferencia**  
Si la diferencia fuese de solo unas décimas, ¿te parecería pedagógicamente relevante o insuficiente? Explica qué información adicional querrías antes de decidir.

## 7. Resultados en lenguaje claro

Una parte clave del trabajo aplicado consiste en traducir tablas y gráficos a un mensaje breve, prudente y útil para docentes o equipos clínicos.

**Microtarea 1. Anticipación**  
Antes de ejecutar la celda, intenta formular en una frase qué dirías si tuvieras que presentar el hallazgo principal en menos de 30 segundos.

In [ ]:
grupo_alta = df['alta_valoracion_global'] == 'Alta valoración global'

conf_alta = df.loc[grupo_alta, 'confianza_priorizar'].mean()
conf_no_alta = df.loc[~grupo_alta, 'confianza_priorizar'].mean()
clar_alta = df.loc[grupo_alta, 'claridad_razonamiento'].mean()
clar_no_alta = df.loc[~grupo_alta, 'claridad_razonamiento'].mean()

solapamiento_conf = (df.loc[~grupo_alta, 'confianza_priorizar'] >= 4).mean() * 100
solapamiento_clar = (df.loc[~grupo_alta, 'claridad_razonamiento'] >= 4).mean() * 100

mensaje = f"""
### Lectura breve de los resultados

- En estos datos sintéticos, el grupo con **alta valoración global** muestra una **confianza media** de **{conf_alta:.2f}** frente a **{conf_no_alta:.2f}** en el grupo con valoración no alta.
- La **claridad media del razonamiento** también es mayor: **{clar_alta:.2f}** frente a **{clar_no_alta:.2f}**.
- La diferencia es visible, pero no perfecta: aproximadamente un **{solapamiento_conf:.0f}%** del grupo con valoración no alta aún informa confianza alta (4 o 5), y un **{solapamiento_clar:.0f}%** mantiene claridad alta.
- La lectura adecuada es de **asociación**: valorar bien el proceso completo aparece relacionado con mejores percepciones sobre la priorización, pero aquí no se demuestra que un momento cause ese resultado.
"""

display(Markdown(mensaje))

**Microtarea 2. Interpretación**  
Señala qué frase del resumen evita una interpretación causal excesiva y por qué es importante mantenerla.

**Microtarea 3. Crítica o transferencia**  
Si tuvieras que compartir este resultado con profesorado de enfermería, ¿qué matiz incluirías para que el mensaje sea útil sin simplificar demasiado?

## 8. Límites del ejercicio

Todo ejercicio de análisis tiene límites. Hacerlos explícitos mejora la calidad de la interpretación y evita convertir una práctica docente en una afirmación demasiado fuerte.

**Microtarea 1. Anticipación**  
Antes de mirar la tabla, piensa qué límite te parece más importante aquí: datos sintéticos, autopercepción inmediata, diseño transversal o simplificación del razonamiento clínico.

In [ ]:
limites = pd.DataFrame(
    {
        'Aspecto': [
            'Origen de los datos',
            'Tipo de medida',
            'Diseño temporal',
            'Complejidad del fenómeno'
        ],
        'Qué permite decir': [
            'Sirven para practicar análisis y lectura de resultados.',
            'Reflejan cómo se perciben confianza y claridad justo después de la simulación.',
            'Permite observar asociaciones en un único momento.',
            'Ayuda a simplificar el problema para aprender el flujo básico de análisis.'
        ],
        'Qué no permite decir': [
            'No representan una cohorte real ni permiten conclusiones empíricas.',
            'No equivalen a aprendizaje consolidado ni a desempeño clínico objetivo.',
            'No permite afirmar que un componente cause mejora.',
            'No agota todo lo que implica el razonamiento clínico al priorizar cuidados.'
        ]
    }
)

display(limites)

**Microtarea 2. Interpretación**  
Elige un límite de la tabla y explica cómo cambiaría tu forma de presentar los resultados si tuvieras que exponerlos públicamente.

**Microtarea 3. Crítica o transferencia**  
¿Qué riesgo ves como más probable en contextos profesionales: sobrerinterpretar la asociación o despreciar demasiado rápido un resultado útil?

## 9. Del ejercicio al estudio real

El último paso es traducir este tutorial a una decisión de investigación viable. El objetivo no es complicar el diseño, sino pasar de una práctica segura con datos sintéticos a un estudio realista y éticamente cuidadoso.

**Microtarea 1. Anticipación**  
Antes de ver la propuesta, piensa cuál sería el siguiente paso más sensato: mejorar el cuestionario, recoger una cohorte real o añadir una parte cualitativa.

In [ ]:
pasos_reales = pd.DataFrame(
    {
        'Paso siguiente': [
            'Diseñar un cuestionario breve y claro',
            'Recoger datos reales en una cohorte concreta',
            'Mantener variables de contexto básicas',
            'Añadir una parte cualitativa breve',
            'Cuidar ética e interpretación'
        ],
        'Aplicación al estudio real': [
            'Usar ítems cortos sobre escenario, repetición, debriefing, confianza y claridad.',
            'Aplicar el cuestionario justo al final de una sesión de simulación.',
            'Registrar experiencia previa y rendimiento percibido para contextualizar resultados.',
            'Complementar con una pregunta abierta o un grupo focal corto sobre cómo articulan prioridades.',
            'Obtener consentimiento, no evaluar rendimiento individual y evitar afirmaciones causales.'
        ]
    }
)

display(pasos_reales)

**Microtarea 2. Interpretación**  
Mira la hoja de ruta y decide qué paso parece más urgente para pasar de un ejercicio didáctico a un estudio defendible.

**Microtarea 3. Crítica o transferencia**  
Propón un siguiente paso concreto para tu contexto: una pregunta de cuestionario, una mejora organizativa o una forma breve de recoger datos cualitativos.